# Practice guide 04_01: static malware detection from PE headers and API fingerprints

**Goal.** Build the static detector yourself: the evaluation metrics of an imbalanced scanner (AUC, average
precision, precision/recall at an operating point), the benign-downsampling experiment, a two-stage
hierarchy over 7 malware families, and the malleable-field preview that class 04_03 exploits. Every `TODO`
cell must be completed; the `check` cells tell you whether you got it right.

Read `guide_04_01.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from scipy import sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. The PE-header data (given)

19,611 files x 77 numeric header fields, 74.4% malware — the *reverse* of a real scanner queue, which we
exploit in part C.

In [ ]:
X, y, cols = mw.load_pe()
print(f"PE data: {X.shape[0]:,} files x {X.shape[1]} fields, malware {y.mean():.2%}")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

**Question A1.** Why does `stratify=y` matter here less than in a 1%-malware problem — and when would it
matter at all?

**Answers A1.** With 74.4% malware a random split almost never moves the class share by much, so stratification
is a cheap safety rather than a necessity. It matters when the rare class is small in absolute count (say 30
benign files): one file moved between train and test changes the class share by 3 points, and with tiny test
counts every mislabelled file moves precision visibly. It also matters for reproducibility: the same strata
make runs comparable.

## Part B. Ranking metrics from scratch

A scanner outputs a score; the analyst picks a threshold. Implement, from the sorted scores alone:

* `auc_score(y, s)`: the probability that a random malware file scores above a random benign one (handle
  ties with 0.5);
* `average_precision(y, s)`: precision summed at the rank of each positive, divided by the number of
  positives;
* `pr_at_threshold(y, s, t)`: precision and recall at threshold `t`.

No scikit-learn in your implementation — the check cells compare with it.

In [ ]:
def rank_matrix(y, s):
    """Sort by score descending; return the sorted labels and the scores."""
    o = np.argsort(s)[::-1]
    return y[o], s[o]


def auc_score(y, s):
    pos, neg = np.sort(s[y == 1]), np.sort(s[y == 0])
    gt = np.searchsorted(neg, pos, side="right").sum()  # pos strictly above neg
    eq = np.searchsorted(neg, pos, side="left").sum() - gt
    return (gt + 0.5 * eq) / (len(pos) * len(neg))


def average_precision(y, s):
    yy, _ = rank_matrix(y, s)
    c = np.cumsum(yy)
    ranks = np.arange(1, len(yy) + 1)
    prec = c / ranks
    return float(prec[yy == 1].mean())


def pr_at_threshold(y, s, t):
    pred = s >= t
    tp = int((pred & (y == 1)).sum())
    p = tp / int(pred.sum()) if pred.sum() else 0.0
    r = tp / int((y == 1).sum())
    return p, r

In [ ]:
# check: a perfect and a random score, then the scikit-learn twins
from sklearn.metrics import average_precision_score, roc_auc_score

rng = np.random.default_rng(0)
yp = np.concatenate([np.ones(50), np.zeros(50)])
assert auc_score(yp, yp) == 1.0 and abs(auc_score(yp, rng.random(100)) - 0.5) < 0.15
s = rng.random(100) * np.where(yp, 1, 0.6)
assert abs(auc_score(yp, s) - roc_auc_score(yp, s)) < 1e-12
assert abs(average_precision(yp, s) - average_precision_score(yp, s)) < 1e-12
print(f"AUC {auc_score(yp, s):.3f}  AP {average_precision(yp, s):.3f}  metrics OK")

**Question B1.** A scanner flags a file when its score is above the threshold. With 74.4% malware in the
test set, what is the precision of a scanner that flags *every* file? And with the realistic 1% base rate?
Which quantity does the base rate enter directly?

**Answers B1.** Flagging everything gives precision = base rate: 0.744 on this benchmark, but 0.01 at a
realistic 1% — one true alarm for every hundred. The base rate enters precision directly (TP/(TP+FP) = b·TPR /
(b·TPR + (1-b)·FPR)); recall and AUC never see it. This is why a 99% TPR detector is unusable at 1% base rate
unless the FPR is far below 1%, and why part D matters.

## Part C. The detector and the imbalance experiment

Train the random forest (given), then downsample the benign *training* files to the fractions 1.0, 0.2, 0.05,
0.01 and measure, for each, the AUC, the AP and the precision at fixed recall 0.95. Fill the table.

In [ ]:
rf_full = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
p_full = rf_full.predict_proba(Xte)[:, 1]
print("full model:", mw.eval_bin(yte, p_full, "RF PE baseline: "))

In [ ]:
idx_b = np.where(ytr == 0)[0]
rowsC = {}
rowsC = {}
rng = np.random.default_rng(0)
idx_m = np.where(ytr == 1)[0]
for frac in [1.0, 0.2, 0.05, 0.01]:
    keep_b = rng.choice(idx_b, size=int(len(idx_b) * frac), replace=False)
    rows = np.concatenate([idx_m, keep_b])
    rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr[rows], ytr[rows])
    p = rf.predict_proba(Xte)[:, 1]
    # smallest threshold with recall >= 0.95: sweep the unique scores descending
    o = np.argsort(p)[::-1]
    yy, ss = yte[o], p[o]
    c = np.cumsum(yy)
    k = int(np.argmax(c >= 0.95 * yy.sum()))  # first rank reaching recall 0.95
    t = ss[k]
    pr, _ = pr_at_threshold(yte, p, t)
    rowsC[frac] = (auc_score(yte, p), average_precision(yte, p), pr)
for frac, (a, ap_, pr) in rowsC.items():
    print(f"benign frac {frac:5.2f}  AUC {a:.4f}  AP {ap_:.4f}  precision@recall.95 {pr:.4f}")

**Question C1.** Which column degrades as the benign fraction drops, and which barely moves? What does that
mean for a vendor who trains on a 1%-malware queue and advertises an AUC?

**Answers C1.** AUC barely moves (0.999 to about 0.985) — ranking is easy; the precision at recall 0.95
collapses as the benign class thins out, because the forest has fewer benign examples to model the FP side.
A vendor trained on an imbalanced queue quoting AUC is hiding the metric that matters: at a real base rate
the false positives dominate the cost, and that is what the precision column measures.

## Part D. API-function fingerprints: flat vs two-stage

The Windows API dataset counts 21,918 API functions per file over 7 classes. Loading it eagerly would need
tens of GB; `malwlib.load_api_sparse` reads 100-row UInt8 slices and keeps a stratified 20,000-file sample
(~6 MB CSR, peak RSS under 0.8 GB). **This cell takes a few minutes the first time; the demo cache makes it
instant afterwards.**

In [ ]:
CACHE = mw.ROOT / ".cache_api_sub"
if CACHE.with_suffix(".npz").exists():
    M, yf = sp.load_npz(CACHE.with_suffix(".npz")), np.load(CACHE.with_suffix(".npy"))
else:
    M, yf = mw.load_api_sparse()
    sp.save_npz(CACHE.with_suffix(".npz"), M)
    np.save(CACHE.with_suffix(".npy"), yf)
print(f"API matrix: {M.shape[0]:,} x {M.shape[1]:,}, density {M.nnz / (M.shape[0] * M.shape[1]):.4f}")

itr, ite = train_test_split(np.arange(len(yf)), test_size=0.25, stratify=yf, random_state=0)

Train the flat 7-class forest (given) and the two-stage pipeline: stage 1 binary benign-vs-malware, stage 2
family only on the files stage 1 calls malicious. Compare macro-F1 end to end.

In [ ]:
flat = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[itr], yf[itr])
pf = flat.predict(M[ite])
from sklearn.metrics import classification_report, f1_score

print(classification_report(yf[ite], pf, digits=3, zero_division=0))
flat_macro = f1_score(yf[ite], pf, average="macro")

In [ ]:
yb = (yf > 0).astype(int)
hier = {}
rf1 = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[itr], yb[itr])
pred1 = rf1.predict(M[ite])
mal_tr = itr[yb[itr] == 1]
rf2 = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[mal_tr], yf[mal_tr])
sel = pred1 > 0
pred2 = np.where(sel, 0, 0)  # start all-benign
fam2 = rf2.predict(M[ite[sel]])
end = np.zeros(len(yf[ite]), dtype=int)
end[sel] = fam2
stage2_rows = yf[ite][sel] > 0
hier = (
    float((pred1[yb[ite] == 1] == 1).mean()),
    f1_score(yf[ite][sel][stage2_rows], fam2[stage2_rows], average="macro"),
    f1_score(yf[ite], end, average="macro"),
)
print(f"flat macro-F1 {flat_macro:.4f}")
print(f"hier: stage-1 recall {hier[0]:.4f}, stage-2 macro-F1 {hier[1]:.4f}, end-to-end macro-F1 {hier[2]:.4f}")

**Question D1.** The hierarchy wins (or loses) only a little macro-F1. Why does the *recall of stage 1* cap
the whole pipeline, and why can stage 2 never recover a file stage 1 missed?

**Answers D1.** A benign prediction short-circuits stage 2, so a stage-1 miss is a permanent zero for the
true family; stage 2 only refines files already called malicious, and inherits stage 1's false positives as
extra (wrong-family) inputs. The end-to-end macro-F1 is therefore bounded by stage-1 recall times whatever
stage 2 achieves. The hierarchy buys *error locality* (family mistakes need the file to pass stage 1 first)
at the price of a single point of failure.

## Part E. What the model looks at — and the preview of 04_03

Print the 8 most important header fields. `malwlib.MALLEABLE` lists the fields an attacker can rewrite
without breaking the program: count how many of the top-20 fields are malleable.

In [ ]:
imp = rf_full.feature_importances_
top = np.argsort(imp)[::-1]
for i in top[:8]:
    print(f"{cols[i]:34s} {imp[i]:.3f}")
n_mal = sum(cols[i] in mw.MALLEABLE for i in top[:20])
print(f"malleable fields among the top 20: {n_mal}")

**Question E1.** Class 04_03 rewrites exactly these fields to the benign median and the detector folds. Two
defences come to mind that do **not** need retraining — name them and say why each is unsatisfying.

**Answers E1.** (1) Drop the malleable fields from the model — but the top fields are version/size indicators
that carry real signal, so the detector gets worse for honest files too. (2) Require a cryptographic
signature (Authenticode) — legitimate for signed vendors only; most malware-bearing channels (fresh builds,
repacked binaries) are unsigned, so coverage collapses. Neither fixes the root cause: the model trusts
metadata the attacker controls, which is why 04_03 ends with adversarial retraining instead.

## Part G. A safe replicator — what the detector cannot even see

`malwlib.replicate` writes the source of a minimal self-copying program, generation by generation,
into a scratch directory. It carries no payload (it only writes the next generation and stops at a
hard cap), is never executed as a subprocess, and touches nothing outside `workdir`. Build 5
generations and read generation 0.

In [ ]:
import tempfile

workdir_g = Path(tempfile.mkdtemp(prefix="guide_04_01_replicator_"))
paths_g = mw.replicate(workdir_g, cap=5)
print(f"{len(paths_g)} generations written to {workdir_g}")
print(paths_g[0].read_text())

**Question G1.** Could you build a 77-column row for this file the way `load_pe` does for the PE
dataset? Could a static detector trained on PE headers, or on API-import counts, return *any* score
for it? What specifically is missing — the schema, or the training data?

**Answers G1.** No: `load_pe`'s 77 columns are PE header fields (`SizeOfCode`, `MajorLinkerVersion`,
...); this file is a `.py` text script with no PE header at all, so the extractor that builds those
columns has nothing to read — not a bad score, no row. The same holds for the API-import matrix: the
columns are specific Windows API names, and a plain file write is not an "import". It is the **schema**
that is missing, not the training data: no amount of labelled examples teaches a PE-header classifier
to score a file format it has no columns for. Self-replication is a *behaviour* (the program writes a
new file, visible as a file-system event at run time), and nothing in a static feature table records
behaviour — that gap is exactly what class 04_02's dynamic, sequential features are built to close.

In [ ]:
import shutil

shutil.rmtree(workdir_g)  # clean up: this demo is confined to its own temp directory

# Reference solution

The solved notebook, with the answers to the questions and the numbers of a full run, is `solution/guide_04_01.ipynb`.